# 02 — Embedding Analysis

Visualize and compare different embedding types using dimensionality reduction.

In [ ]:
import numpy as np
import json
from pathlib import Path
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

EMB_DIR = Path("../data/embeddings/bge-large-en-v1.5/")
if not EMB_DIR.exists():
    EMB_DIR = Path("../code/embedding_generator/output/")

profile_emb = np.load(EMB_DIR / "profile_embeddings.npy")
genome_emb = np.load(EMB_DIR / "genome_embeddings.npy")
bert_emb = np.load(EMB_DIR / "bert_title_embeddings.npy")

print(f"Profile embeddings: {profile_emb.shape}")
print(f"Genome embeddings: {genome_emb.shape}")
print(f"BERT title embeddings: {bert_emb.shape}")

## t-SNE Comparison: LLM Profile vs Genome vs BERT

Sample 2000 movies for visualization.

In [ ]:
np.random.seed(42)
sample_idx = np.random.choice(len(profile_emb), 2000, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, emb, title in zip(axes, 
    [profile_emb[sample_idx], genome_emb[sample_idx], bert_emb[sample_idx]],
    ["LLM Profile (1024d)", "Genome PCA (128d)", "BERT Title (1024d)"]):
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    coords = tsne.fit_transform(emb)
    ax.scatter(coords[:, 0], coords[:, 1], s=1, alpha=0.3)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])

plt.suptitle("t-SNE Visualization of Movie Embeddings (n=2000)", y=1.02)
plt.tight_layout()
plt.show()

## Cosine Similarity Distributions

Compare how discriminative each embedding type is.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

n_sample = 500
idx = np.random.choice(len(profile_emb), n_sample, replace=False)

fig, ax = plt.subplots(figsize=(10, 5))

for emb, label, color in [
    (profile_emb[idx], "LLM Profile", "blue"),
    (genome_emb[idx], "Genome PCA", "orange"),
    (bert_emb[idx], "BERT Title", "green"),
]:
    sim = cosine_similarity(emb)
    upper = sim[np.triu_indices(n_sample, k=1)]
    ax.hist(upper, bins=100, alpha=0.5, label=f"{label} (mean={upper.mean():.3f})", color=color, density=True)

ax.set_xlabel("Cosine Similarity")
ax.set_ylabel("Density")
ax.set_title("Pairwise Cosine Similarity Distribution")
ax.legend()
plt.tight_layout()
plt.show()